In [37]:
import pandas as pd

In [38]:
df = pd.read_csv(r"C:\Users\hp\Desktop\Amazon\CSV Files\amazon_india_2017.csv")

In [ ]:
df['order_date']

In [3]:
df.shape

(77385, 34)

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77385 entries, 0 to 77384
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          77385 non-null  object 
 1   order_date              77385 non-null  object 
 2   customer_id             77385 non-null  object 
 3   product_id              77385 non-null  object 
 4   product_name            77385 non-null  object 
 5   category                77385 non-null  object 
 6   subcategory             77385 non-null  object 
 7   brand                   77385 non-null  object 
 8   original_price_inr      77385 non-null  object 
 9   discount_percent        77385 non-null  float64
 10  discounted_price_inr    77385 non-null  float64
 11  quantity                77385 non-null  int64  
 12  subtotal_inr            77385 non-null  float64
 13  delivery_charges        71196 non-null  float64
 14  final_amount_inr        77385 non-null

In [41]:
import numpy as np

df.replace("", np.nan, inplace=True)


In [42]:
dfc = df.copy()

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [69]:
import pandas as pd

dfc['order_date'] = (
    dfc['order_date']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', '', regex=True)
)

dfc['order_date'] = pd.to_datetime(
    dfc['order_date'],
    dayfirst=True,
    errors='coerce'
)
dfc['order_date'] = dfc['order_date'].dt.strftime('%Y-%m-%d')


C:\Users\hp\AppData\Local\Temp\ipykernel_9796\912995083.py:12: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dfc['order_date'] = pd.to_datetime(


In [71]:
dfc['order_date'].head(10)

0    2017-01-17
1    2017-01-28
2    2017-01-30
3    2017-01-04
4    2017-01-07
5    2017-01-14
6           NaN
7    2017-01-19
8    2017-01-24
9    2017-01-13
Name: order_date, dtype: object

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees.

In [45]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
        .astype(str)                      
        .str.replace('₹', '', regex=False) 
        .str.replace(',', '', regex=False)
        .str.replace('Rs ', '', regex=False)
        .str.strip()                
)

dfc['original_price_inr'] = pd.to_numeric(
    dfc['original_price_inr']
)


Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.

In [46]:
import re

def parse_rating(r):
    if pd.isna(r):
        return np.nan

    if '/' in r:
        a, b = r.split('/')
        return float(a) / float(b) * 5

    m = re.search(r'\d+\.?\d*', r)
    return float(m.group()) if m else np.nan


dfc['customer_rating'] = dfc['customer_rating'].apply(parse_rating)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.

In [47]:
dfc['customer_city'] = (
    dfc['customer_city']
    .str.lower()
    .str.strip()
)
 
city_map = {
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'bangalore/bengaluru': 'Bengaluru',
    'bengalore' : 'Bengaluru',
    'Bengaluru' : 'banglore',
    
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'mumbai/bombay': 'Mumbai',
    'mumba' : 'Mumbai',
    'calcutta' : 'kolkata',

    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi/new delhi': 'Delhi',
    'delhi NCR' : 'Delhi',
    'delhi ncr' : 'Delhi',

    'chenai' : 'chennai',
    'madras' : 'chennai'
}

dfc['customer_city'] = dfc['customer_city'].replace(city_map)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [50]:
import numpy as np

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    dfc[col] = dfc[col].replace(['', ' ', 'NA', 'N/A', None, 'None'], np.nan)


bool_map = {
    True: True,
    'True': True,
    'true': True,
    'Yes': True,
    'yes': True,
    'Y': True,
    'y': True,
     1: True,
    
     False: False,
    'False': False,
    'false': False,
    'No': False,
    'no': False,
    'N': False,
    'n': False,
     0: False
}


for col in bool_cols:
    dfc[col] = dfc[col].map(bool_map)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.

In [49]:
dfc.columns = dfc.columns.str.strip()

category_map = {
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics',
    'Electronicss': 'Electronics',
    'Electronics & Accessories': 'Electronics',
    'Electronic': 'Electronics',
    'clothing': 'Fashion'
}

dfc['category'] = dfc['category'].replace(category_map)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [51]:
import numpy as np

days_map = {
    'Express': '0',
    'Same Day': '0',
    '-1': 'None',
    '1-2 days': '2'
}

dfc['delivery_days'] = dfc['delivery_days'].replace(days_map)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [53]:
dup_cols = [
    "customer_id",
    "product_id",
    "order_date",
    "final_amount_inr"
]

price_cols = [
    "original_price_inr",
    "discounted_price_inr",
    "subtotal_inr",
    "final_amount_inr"
]

dfc["dup_count"] = (
    dfc.groupby(dup_cols)["transaction_id"]
      .transform("count")
)


dfc["price_identical"] = (
    dfc.groupby(dup_cols)[price_cols]
      .transform("nunique")
      .max(axis=1) == 1
)


dfc["is_high_value"] = dfc["final_amount_inr"] > 5000
dfc["is_bulk_customer"] = dfc["customer_spending_tier"].isin(["Premium"])
dfc["is_bulk_quantity"] = dfc["quantity"] > 1

dfc["is_duplicate_candidate"] = dfc["dup_count"] > 1


In [54]:
df_deduped = dfc[dfc["is_duplicate_candidate"]].drop_duplicates(subset=dup_cols, keep="first")


In [ ]:
print("Rows deleted:", (df_deduped))

In [56]:
len(dfc)

77385

In [57]:
cols_to_drop = [
    "dup_count",
    "price_identical",
    "is_high_value",
    "is_bulk_customer",
    "is_bulk_quantity",
    "is_duplicate_candidate"
]

dfc = dfc.drop(columns=cols_to_drop)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [58]:
import numpy as np

dfc["product_median_price"] = (
    dfc.groupby("product_id")["final_amount_inr"]
      .transform("median")
)

dfc["price_outlier"] = (
    dfc["final_amount_inr"] > 50 * dfc["product_median_price"]
)

dfc.loc[dfc["price_outlier"], "final_amount_inr"] /= 100
dfc.loc[dfc["price_outlier"], "discounted_price_inr"] /= 100
dfc.loc[dfc["price_outlier"], "original_price_inr"] /= 100

dfc["subtotal_inr"] = dfc["discounted_price_inr"] * dfc["quantity"]
dfc["final_amount_inr"] = dfc["subtotal_inr"] + dfc["delivery_charges"].fillna(0)

dfc["price_corrected_flag"] = dfc["price_outlier"]

dfc.drop(columns=["product_median_price"], inplace=True)

corrected_rows = dfc[dfc["price_corrected_flag"]]

print(corrected_rows)


Empty DataFrame
Columns: [transaction_id, order_date, customer_id, product_id, product_name, category, subcategory, brand, original_price_inr, discount_percent, discounted_price_inr, quantity, subtotal_inr, delivery_charges, final_amount_inr, customer_city, customer_state, customer_tier, customer_spending_tier, customer_age_group, payment_method, delivery_days, delivery_type, is_prime_member, is_festival_sale, festival_name, customer_rating, return_status, order_month, order_year, order_quarter, product_weight_kg, is_prime_eligible, product_rating, price_outlier, price_corrected_flag]
Index: []

[0 rows x 36 columns]


In [59]:
dfc = dfc.drop(
    columns=[
        "price_outlier",
        "price_corrected_flag"       
    ]
)



Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [60]:
dfc["payment_method"] = (
    dfc["payment_method"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

payment_map = {
    "UPI": "UPI",
    "PHONEPE": "UPI",
    "GOOGLEPAY": "UPI",
    "GPAY": "UPI",
    "PAYTM": "UPI",

    "CREDIT CARD": "Credit Card",
    "CREDIT_CARD": "Credit Card",
    "CC": "Credit Card",

    "DEBIT CARD": "Debit Card",
    "DC": "Debit Card",

    "COD": "Cash on Delivery",
    "CASH ON DELIVERY": "Cash on Delivery",

    "NET BANKING": "Net Banking"
}

dfc["payment_method"] = dfc["payment_method"].replace(payment_map)

In [77]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
    .astype(str)
    .str.replace('-', '', regex=False)
    .astype(float)
)


In [61]:
dfc["payment_method"].unique()

array(['Cash on Delivery', 'Debit Card', 'UPI', 'Credit Card',
       'Net Banking'], dtype=object)

In [76]:
import pandas as pd

columns = [
    "transaction_id","order_date","customer_id","product_id","product_name",
    "category","subcategory","brand","original_price_inr","discount_percent",
    "discounted_price_inr","quantity","subtotal_inr"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

transaction_id: ['TXN_2017_00000001', 'TXN_2017_00000002', 'TXN_2017_00000003', 'TXN_2017_00000004', 'TXN_2017_00000005', 'TXN_2017_00000006', 'TXN_2017_00000007', 'TXN_2017_00000008', 'TXN_2017_00000009', 'TXN_2017_00000010', 'TXN_2017_00000011', 'TXN_2017_00000012', 'TXN_2017_00000013', 'TXN_2017_00000014', 'TXN_2017_00000015', 'TXN_2017_00000016', 'TXN_2017_00000017', 'TXN_2017_00000018', 'TXN_2017_00000019', 'TXN_2017_00000020', 'TXN_2017_00000021', 'TXN_2017_00000022', 'TXN_2017_00000023', 'TXN_2017_00000024', 'TXN_2017_00000025', 'TXN_2017_00000026', 'TXN_2017_00000027', 'TXN_2017_00000028', 'TXN_2017_00000029', 'TXN_2017_00000030', 'TXN_2017_00000031', 'TXN_2017_00000032', 'TXN_2017_00000033', 'TXN_2017_00000034', 'TXN_2017_00000035', 'TXN_2017_00000036', 'TXN_2017_00000037', 'TXN_2017_00000038', 'TXN_2017_00000039', 'TXN_2017_00000040', 'TXN_2017_00000041', 'TXN_2017_00000042', 'TXN_2017_00000043', 'TXN_2017_00000044', 'TXN_2017_00000045', 'TXN_2

In [28]:
import pandas as pd

columns = [
    "delivery_charges",
    "final_amount_inr","customer_city","customer_state","customer_tier",
    "customer_spending_tier","customer_age_group","payment_method","delivery_days",
    "delivery_type","is_prime_member","is_festival_sale",
    "festival_name"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

delivery_charges: ['0.0']

final_amount_inr: ['10000.22', '100002.3', '100007.92', '100021.89', '100049.34', '100049.6', '10006.55', '100060.14', '100062.01', '100062.3', '10007.46', '100071.62', '100074.38', '10008.52', '100086.12', '100094.13', '100100.02', '10011.78', '100119.21', '10012.93', '10013.08', '100130.17', '100133.9', '100139.28', '10014.04', '10014.11', '100151.01', '10016.7', '100160.17', '100164.99', '100165.12', '100166.97', '10018.8', '10019.53', '100190.24', '100197.28', '1002.56', '100210.32', '100217.16', '100218.69', '10022.68', '10023.17', '100231.51', '100236.34', '10024.74', '100245.55', '10025.02', '10027.05', '100272.19', '100277.18', '10028.35', '100280.6', '100288.27', '100296.04', '100297.19', '100298.26', '100310.37', '100316.4', '10033.27', '10033.51', '100337.97', '100340.16', '100350.39', '100353.11', '100358.70000000001', '100362.19', '10037.74', '10038.52', '10039.22', '10039.58', '10039.78', '100392.17', '10040.79', 

In [30]:
import pandas as pd

columns = [    "customer_rating","return_status","order_month","order_year","order_quarter",
    "product_weight_kg","is_prime_eligible","product_rating"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

customer_rating: ['3.0', '3.5', '4.0', '4.5', '5.0']

return_status: ['Cancelled', 'Delivered', 'Returned']

order_month: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

order_year: ['2017']

order_quarter: ['1', '2', '3', '4']

product_weight_kg: ['0.03', '0.04', '0.05', '0.06', '0.07', '0.08', '0.1', '0.12', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2', '0.21', '0.22', '0.23', '0.24', '0.25', '0.29', '0.3', '0.32', '0.33', '0.34', '0.35', '0.4', '0.42', '0.43', '0.45', '0.46', '0.47', '0.48', '0.49', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.62', '0.63', '0.64', '0.65', '0.66', '0.68', '0.69', '0.72', '0.73', '0.75', '0.78', '1.2', '1.21', '1.29', '1.39', '1.4', '1.46', '1.5', '1.62', '1.74', '1.76', '1.79', '1.81', '1.98', '1.99', '2.01', '2.02', '2.04', '2.06', '2.17', '2.18', '2.26', '2.27', '2.33', '2.37', '2.42', '2.49', '2.57', '2.6', '2.66', '2.68', '2.69', '2.7', '2.75', '2.8', '21.84', '24.86'

In [ ]:
dfc

In [34]:
print(len(dfc.columns))
print(dfc.columns.tolist())


34
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']


In [35]:
dfc[dfc.duplicated()]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


In [75]:
dfc["original_price_inr"].head(100)

0      25367.39
1      16997.28
2      47409.63
3      45555.75
4     100190.24
        ...    
95     93996.79
96     14088.24
97     25367.39
98     49201.00
99     28625.61
Name: original_price_inr, Length: 100, dtype: float64

In [81]:
import pandas as pd

decimal_cols = dfc.select_dtypes(include=['float', 'float64']).columns

dfc[decimal_cols] = dfc[decimal_cols].round(2)
print(decimal_cols)


Index(['original_price_inr', 'discount_percent', 'discounted_price_inr',
       'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_rating', 'product_weight_kg', 'product_rating'],
      dtype='object')


In [82]:
dfc.to_csv(r"C:\Users\hp\Desktop\Amazon\CSV_Clean_Files\amazon_india_2017_clean.csv",header='infer',index=False)